In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# specify parameters
pipeline_params = {
    "zone": "test_archivestore"
}
step_params={
    "X": "something_else",
    "step_name": "sinara_quick_test"
}
substep_params={
    "param1":"None1",
    "param2":"None2"}

In [3]:
# define substep interface
from sinara.substep import NotebookSubstep, default_param_values, ENV_NAME, PIPELINE_NAME, ZONE_NAME, STEP_NAME, RUN_ID, ENTITY_NAME, ENTITY_PATH, SUBSTEP_NAME

substep = NotebookSubstep(pipeline_params, step_params, substep_params, **default_param_values("params/test_tmp_inout_params.json"))

substep.interface(
    tmp_entities =
    [
        { ENTITY_NAME: "tmp_dir_to_store" },
        
        { ENTITY_NAME: "tmp_dir_to_load" }
    ],
    
    outputs = 
    [
        { ENTITY_NAME: "stored_files" }
    ]

)

substep.print_interface_info()

substep.exit_in_visualize_mode()

**OUTPUTS:**


[{'user.pipeline.zone.sinara_quick_test.stored_files': '/data/home/jovyan/pipeline/zone/sinara_quick_test/run-23-11-17-043546/stored_files'}]




**TMP ENTITIES:**


[{'tmp:user.pipeline.zone.sinara_quick_test.tmp_dir_to_store': '/data/tmp/user/pipeline/zone/sinara_quick_test/run-23-11-17-043546/tmp_dir_to_store'},
 {'tmp:user.pipeline.zone.sinara_quick_test.tmp_dir_to_load': '/data/tmp/user/pipeline/zone/sinara_quick_test/run-23-11-17-043546/tmp_dir_to_load'}]




In [4]:
from sinara.spark import SinaraSpark

spark = SinaraSpark.run_session(0)
SinaraSpark.ui_url()

Session is run


'http://localhost:4040'

In [5]:
# write tmp outputs
tmp_entities = substep.tmp_entities()

for i in range(1, 10):
    with open(f'{tmp_entities.tmp_dir_to_store}/file{i}.txt', 'w') as f:
        f.write(str(i))

In [6]:
from sinara.archivestore import SinaraArchiveStore
arhivestore = SinaraArchiveStore(spark)

In [7]:
import glob
import os

outputs = substep.outputs()

arhivestore.pack_files_from_tmp_to_store(tmp_entities.tmp_dir_to_store, outputs.stored_files)
stored_files = [os.path.basename(x) for x in glob.glob(tmp_entities.tmp_dir_to_store + '/*')]
print(stored_files)

['file7.txt', 'file8.txt', 'file1.txt', 'file9.txt', 'file2.txt', 'file4.txt', 'file3.txt', 'file5.txt', 'file6.txt']


In [8]:
arhivestore.unpack_files_from_store_to_tmp(outputs.stored_files, tmp_entities.tmp_dir_to_load)
loaded_files = [os.path.basename(x) for x in glob.glob(tmp_entities.tmp_dir_to_load + '/*')]
print(loaded_files)

['file7.txt', 'file8.txt', 'file1.txt', 'file9.txt', 'file2.txt', 'file4.txt', 'file3.txt', 'file5.txt', 'file6.txt']


In [9]:
assert set(stored_files) == set(loaded_files), "stored and loaded files are not equal"

In [10]:
SinaraSpark.stop_session()